In [0]:
# ============================================
# CityFlow — 01_bronze_ingestion
# ============================================
import urllib.parse

# Secure credentials — from Databricks Secrets
ACCESS_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="aws-access-key")
SECRET_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="aws-secret-key")
encoded_secret = urllib.parse.quote(SECRET_KEY, safe="")

S3_BUCKET = "cityflow-taxi-data"
RAW_PATH = f"s3a://{ACCESS_KEY}:{encoded_secret}@{S3_BUCKET}/raw"
BRONZE_PATH = f"s3a://{ACCESS_KEY}:{encoded_secret}@{S3_BUCKET}/bronze"

# Load CSV
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{RAW_PATH}/taxi/yellow_tripdata_2016-01.csv")

print(f"Total rows: {df.count()}")
df.show(5)

Total rows: 10906858
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RatecodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2016-01-01 00:00:00|  2016-01-01 00:00:00|              2|          1.1|-73.99037170410156| 40.734695434

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Metadata columns 
df_bronze = df \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path"))

# Delta format — secure path
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE_PATH}/taxi/")

print("Bronze layer saved!")
print(f"Total records: {df_bronze.count()}")

Bronze layer saved!
Total records: 10906858
